# 155 — MLP Feature Attribution

MLPs are the knowledge stores of transformers. This module provides fine-grained
tools for understanding what individual neurons contribute to predictions:
which neurons promote or suppress tokens, their activation profiles, and
how neurons cooperate.

## Why This Matters

MLP feature attribution provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_feature_attribution import (
    neuron_logit_attribution,
    active_neuron_profile,
    mlp_layer_attribution,
    neuron_feature_directions,
    neuron_cooperation,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(15)

## Neuron Logit Attribution

Which neurons in a given MLP layer promote or suppress the predicted token?
Each neuron's contribution = activation * (W_out row projected onto unembed direction).

In [ ]:
result = neuron_logit_attribution(model, tokens, layer=0, top_k=5)

print(f"Layer {result['layer']}, target token: {result['target_token']}")
print(f"Total MLP logit: {result['total_mlp_logit']:.4f}")
print(f"Top-5 fraction: {result['top_k_fraction']:.2%}\n")

print("Promoting neurons:")
for n in result['promoting']:
    print(f"  neuron {n['neuron']:3d}: act={n['activation']:+.4f} * dir={n['direction_alignment']:+.4f} "
          f"= {n['logit_contribution']:+.4f}")

print("\nSuppressing neurons:")
for n in result['suppressing']:
    print(f"  neuron {n['neuron']:3d}: act={n['activation']:+.4f} * dir={n['direction_alignment']:+.4f} "
          f"= {n['logit_contribution']:+.4f}")

## Active Neuron Profile

How sparse is the MLP? How many neurons are active, dead, or always on?

In [ ]:
for layer in range(4):
    result = active_neuron_profile(model, tokens, layer=layer)
    print(f"Layer {result['layer']}: d_mlp={result['d_mlp']}  "
          f"active/pos={result['mean_active_per_position']:.0f}  "
          f"sparsity={result['sparsity']:.3f}  "
          f"dead={result['never_active']}  always_on={result['always_active']}")

## MLP Layer Attribution

Compare how much each MLP layer contributes to the target logit.

In [ ]:
result = mlp_layer_attribution(model, tokens)

print(f"Target token: {result['target_token']}")
print(f"Total MLP logit: {result['total_mlp_logit']:.4f}")
print(f"Most promoting: layer {result['most_promoting_layer']}")
print(f"Most suppressing: layer {result['most_suppressing_layer']}\n")

for p in result['per_layer']:
    direction = '+' if p['promotes'] else '-'
    bar = '#' * int(abs(p['logit_contribution']) * 20)
    print(f"Layer {p['layer']}: logit={p['logit_contribution']:+.4f}  "
          f"norm={p['output_norm']:.4f}  {direction}{bar}")

## Neuron Feature Directions

What vocabulary items does each neuron promote when it fires?
This looks at the static W_out directions projected through W_U.

In [ ]:
result = neuron_feature_directions(model, layer=0, top_k=3)

print(f"Layer {result['layer']}: mean_norm={result['mean_output_norm']:.4f}  "
      f"max_norm={result['max_output_norm']:.4f}\n")

for n in result['per_neuron']:
    print(f"Neuron {n['neuron']}: norm={n['output_norm']:.4f}")
    print(f"  Promotes tokens: {n['top_promoted_tokens']} (max={n['max_promotion']:.4f})")
    print(f"  Suppresses tokens: {n['top_suppressed_tokens']} (max={n['max_suppression']:.4f})")

## Neuron Cooperation

Find neurons that fire together and write in the same direction (cooperating)
or opposing directions (competing).

In [ ]:
result = neuron_cooperation(model, tokens, layer=0, top_k=5)

print(f"Layer {result['layer']}: {result['n_active']} active neurons\n")

print("Cooperating pairs:")
for p in result['cooperating_pairs']:
    print(f"  N{p['neuron_a']}-N{p['neuron_b']}: cos={p['cosine_similarity']:.3f}  "
          f"combined_norm={p['combined_norm']:.4f}")

print("\nCompeting pairs:")
for p in result['competing_pairs']:
    print(f"  N{p['neuron_a']}-N{p['neuron_b']}: cos={p['cosine_similarity']:.3f}  "
          f"combined_norm={p['combined_norm']:.4f}")